# TP5 : sécuriser l'accès au cluster

| Repère | Valeur |
|---|---|
| Moment | Jour 2, 09:50-10:45, après la démo du TP (09:45-09:50) |
| Compétences | C9 Listeners, TLS et SASL/SCRAM ; C10 ACL et quotas |
| Slides de rappel | S43 (vue d'ensemble), S44 (protocoles des listeners), S45 (TLS), S47 (SCRAM en KRaft), S48 (ACL), S49 (quotas) |
| Version Markdown | `enonce.md`, mêmes étapes en ligne de commande seule |

**Mode d'emploi du notebook**

- Exécutez les cellules dans l'ordre avec `Maj+Entrée`. Après un redémarrage du noyau, les variables sont perdues : relancez la cellule d'outils, puis les cellules qui définissent les fonctions et les variables dont vous avez besoin. Ne relancez pas une cellule qui modifie le cluster (création d'un topic, arrêt d'un nœud) : elle échouerait, ou referait l'action.
- Chaque étape suit le même schéma : une explication, un **exemple en ligne de commande**, une **zone `# TODO`**, puis une **vérification**. Les commandes qui modifient le cluster sont affichées sans être exécutées (« ne pas lancer en plus du TODO »).
- Dans ce TP, deux clients Python cohabitent : le client d'administration `admin = AdminClient(CONF)` passe par le listener INTERNAL (`BS`, sans sécurité, réservé à l'administration du lab) ; les clients des comptes `admin`, `alice` et `bob` passent par le listener SECURE (`BS_SECURE`).
- Les fonctions `connection()`, `send()` et `read()` de la cellule suivante sont fournies : elles renvoient un code d'erreur au lieu de laisser la bibliothèque remplir la cellule de messages.

In [ ]:
# Connexion au lab : adresses des brokers, configuration du client Python, dossier de travail
import os
import time

from confluent_kafka.admin import AdminClient

BS = os.environ["BS"]  # listener INTERNAL, comme $BS dans le conteneur tools
BS_SECURE = os.environ["BS_SECURE"]  # listener SECURE (SASL_SSL)
CONF = {"bootstrap.servers": BS, "log_level": 2}

# Les commandes docker compose et les fichiers config/ se lisent depuis le dossier lab/
LAB_DIR = os.environ["FORMATION_DIR"] + "/lab"
os.chdir(LAB_DIR)
os.environ["COMPOSE_PROGRESS"] = "plain"  # docker compose sans animation dans le notebook

In [ ]:
# Deux fonctions d'aide, écrites ici : elles ne font pas partie de confluent-kafka
def wait_for(condition, timeout=60, step=2, message="condition"):
    """Rappelle condition() jusqu'à obtenir une valeur vraie, et la renvoie."""
    start = time.time()
    while True:
        try:
            value = condition()
        except Exception:
            value = None
        if value:
            return value
        if time.time() - start >= timeout:
            raise TimeoutError("%s : non atteinte en %d s" % (message, timeout))
        time.sleep(step)

def table(lines, headers):
    """Affiche des lignes en tableau Markdown dans le notebook."""
    from IPython.display import Markdown, display

    md = ["| " + " | ".join(headers) + " |", "|" + "---|" * len(headers)]
    md += ["| " + " | ".join(str(v).replace("|", "\\|") for v in line) + " |" for line in lines]
    display(Markdown("\n".join(md)))

In [ ]:
import logging
import os
import time

from confluent_kafka import Consumer, KafkaException, Producer
from confluent_kafka.admin import (AclBinding, AclBindingFilter, AclOperation, AclPermissionType, AdminClient,
                                   NewTopic, ResourcePatternType, ResourceType, ScramCredentialInfo,
                                   ScramMechanism, UserScramCredentialUpsertion)

# Certificat de la CA du lab, au format PEM (librdkafka ne lit pas le truststore PKCS12 des clients Java)
CA_PEM = os.path.join(LAB_DIR, "security", "certs", "ca.crt")


class _LogCapture(logging.Handler):
    """Reçoit les messages de librdkafka (niveau INFO et plus) au lieu de les afficher."""

    def __init__(self):
        super().__init__(logging.INFO)
        self.lines = []

    def emit(self, record):
        self.lines.append((record.levelno, record.getMessage().split("]: ", 1)[-1]))


def connection(config):
    """Liste les topics avec config : renvoie ("OK", topics) ou (code d'erreur, premier message d'erreur)."""
    log_capture = _LogCapture()
    logger = logging.getLogger("tp5.connexion.%d" % id(log_capture))
    logger.propagate = False
    logger.setLevel(logging.INFO)
    logger.addHandler(log_capture)
    errors = []
    client = AdminClient({**config, "logger": logger, "error_cb": errors.append})
    try:
        return "OK", sorted(client.list_topics(timeout=10).topics)
    except KafkaException as e:
        client.poll(1)  # error_cb et logger sont servis par poll()
        code = (errors[0] if errors else e.args[0]).name()
        # Le message le plus grave : une déconnexion est journalisée en INFO, un refus d'authentification en ERROR
        lines = sorted(log_capture.lines, key=lambda line: -line[0])
        return code, lines[0][1] if lines else e.args[0].str()


def send(config, topic, value, key=None, **settings):
    """Envoie un message : renvoie None s'il est écrit, sinon le code d'erreur."""
    report = []
    producer = Producer({**config, "message.timeout.ms": 15000, "log_level": 0, **settings})
    try:
        producer.produce(topic, key=key, value=value, on_delivery=lambda err, msg: report.append(err))
        producer.flush(20)
    except Exception as e:
        # Erreur fatale (identifiant de producteur idempotent refusé) : KafkaException,
        # parfois enveloppée par la bibliothèque dans une autre exception
        cause = e if isinstance(e, KafkaException) else e.__cause__
        return cause.args[0].name()
    return report[0].name() if report and report[0] is not None else None


def read(config, group, count=5, delay=15):
    """Lit orders avec le groupe, à partir de ses offsets validés (sinon du début) :
    renvoie les valeurs lues, ou le code d'erreur."""
    # Validation automatique des offsets, comme le consommateur console : le groupe garde sa
    # position (un groupe vide sans offset validé est supprimé par le broker)
    consumer = Consumer({**config, "group.id": group, "auto.offset.reset": "earliest", "log_level": 0})
    consumer.subscribe(["orders"])
    values, end = [], time.time() + delay
    try:
        while len(values) < count and time.time() < end:
            message = consumer.poll(1)
            if message is None:
                continue
            if message.error():
                return message.error().name()
            values.append(message.value().decode())
    finally:
        consumer.close()
    return values


print("INTERNAL :", BS)
print("SECURE   :", BS_SECURE)
print("CA du lab :", CA_PEM, "(présent)" if os.path.exists(CA_PEM) else "(ABSENT : lancer scripts/prepare.sh)")

## 1. Objectifs

À la fin du TP, vous savez :

- lire la configuration de sécurité d'un listener : protocole, mécanisme SASL, magasins TLS, authorizer ;
- créer des comptes SCRAM stockés dans les métadonnées KRaft ;
- configurer un client SASL_SSL, en Java (`.properties`) comme en Python (librdkafka) ;
- distinguer un échec d'authentification d'un refus d'autorisation, côté client et dans `kafka-authorizer.log` ;
- donner le minimum de droits à un producteur et à un consommateur ;
- limiter le débit d'un client par un quota et mesurer le ralentissement.

## 2. Prérequis

- Jour 1 terminé : topic `orders` présent.
- Lab démarré, de préférence en mode `extended` : `scripts/start.sh extended` puis `scripts/validate.sh`.
- Fichiers clients Java dans `/opt/lab/clients/` (conteneur `tools`) ; certificat de la CA en PEM dans `lab/security/certs/ca.crt`.
- Notions : slides S43 à S51.

## 3. Architecture utilisée

```
                 conteneur tools (Java)  et  notebook (Python)
          +---------------------+------------------------------+
          | BS (INTERNAL)       | BS_SECURE (SECURE)           |
          | PLAINTEXT           | SASL_SSL + SCRAM-SHA-512     |
          | principal ANONYMOUS | principal User:<nom du compte>|
          +----------+----------+---------------+--------------+
                     |                          |
                     v                          v
        kafka-N:19092                  kafka-N:19094
        administration du lab          TLS : certificat du broker signé par la CA du lab
                                       SASL : compte SCRAM (stocké dans __cluster_metadata)
                                       Autorisation : StandardAuthorizer + ACL
                                       Quotas : par principal (user) ou client.id
```

| Compte | Mot de passe | Rôle | Droits visés |
|---|---|---|---|
| `admin` | `admin-secret` | Administration | Super-utilisateur |
| `alice` | `alice-secret` | Service Commandes (`client.id=orders-service`) | Écrire dans `orders` |
| `bob` | `bob-secret` | Service Facturation (`client.id=billing-service`) | Lire `orders` avec le groupe `billing-secure` |

**Écart volontaire avec la production** :

| Lab | Production |
|---|---|
| `User:ANONYMOUS` super-utilisateur, pour que le listener INTERNAL reste ouvert aux TP | Tous les listeners authentifiés ; ANONYMOUS n'a aucun droit |
| Mots de passe écrits dans le notebook | Secrets lus dans un coffre ou des variables d'environnement, jamais dans le code |

## 4. Contexte métier

L'audit de sécurité de NovaShop exige que les applications ne se connectent plus sans identité. Chaque service doit avoir son compte, avec les droits strictement nécessaires. Le service Commandes a déjà saturé le cluster lors d'un import massif : il doit être bridé.

## 5. Énoncé

1. Vérifiez comment le listener SECURE est protégé.
2. Créez les comptes `admin`, `alice` et `bob`.
3. Donnez à `alice` le droit d'écrire dans `orders`, et à `bob` celui de lire `orders` avec le groupe `billing-secure` uniquement.
4. Prouvez que chaque refus est bien journalisé.
5. Limitez `alice` à 100 Ko/s en production et mesurez l'effet.

## 6. Étapes

### 6.1 Lire la configuration du listener SECURE

**Exemples en ligne de commande** : la configuration de sécurité de `kafka-1`, puis un client Java sans configuration de sécurité face au listener SECURE (20 s au plus).

In [ ]:
!grep -nE '^(listener.security.protocol.map|listener.name.secure|authorizer|super.users|allow.everyone)' config/kafka/kafka-1/server.properties

In [ ]:
!docker exec kafka-tools bash -c "timeout 20 kafka-topics.sh --bootstrap-server $BS_SECURE --list 2>&1 | grep -E '^Error|OutOfMemory' | sort -u"

In [ ]:
# TODO 6.1 : le même essai avec le client Python
# Appelez connection() avec la configuration minimale {"bootstrap.servers": BS_SECURE} (sans sécurité)
# et rangez le résultat (code, message) dans plaintext_attempt.
plaintext_attempt = None

print(plaintext_attempt)

In [ ]:
# Vérification 6.1
assert plaintext_attempt is not None and plaintext_attempt[0] != "OK", "À revoir : client sans sécurité refusé : %s" % (plaintext_attempt,)
print("OK  client sans sécurité refusé : %s" % (plaintext_attempt,))
assert plaintext_attempt is not None and "APIVERSION_QUERY" in plaintext_attempt[1], "À revoir : le broker coupe la connexion dès la première requête Kafka"
print("OK  le broker coupe la connexion dès la première requête Kafka")

**Résultat attendu** :

- `listener.security.protocol.map` associe `SECURE` à `SASL_SSL` ; `listener.name.secure.sasl.enabled.mechanisms=SCRAM-SHA-512` ;
- magasins PKCS12 : `kafka-1.keystore.p12` (identité du broker), `truststore.p12` (CA du lab) ; `listener.name.secure.ssl.client.auth=none` : pas de certificat client exigé ;
- `authorizer.class.name=org.apache.kafka.metadata.authorizer.StandardAuthorizer`, `super.users=User:admin;User:ANONYMOUS`, `allow.everyone.if.no.acl.found=false` ;
- le client Java échoue avec `The AdminClient thread has exited` et une `java.lang.OutOfMemoryError: Java heap space` ;
- le client Python échoue sans erreur de mémoire : `_ALL_BROKERS_DOWN`, avec `Disconnected: connection closed by peer ... in state APIVERSION_QUERY`.

### 6.2 Créer les comptes SCRAM

Un compte SCRAM est un « credential » attaché à un nom d'utilisateur : mécanisme, nombre d'itérations du hachage et mot de passe. Kafka ne stocke que le sel et les clés dérivées.

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-configs.sh --bootstrap-server $BS --alter --entity-type users --entity-name alice \
  --add-config 'SCRAM-SHA-512=[iterations=8192,password=alice-secret]'
```

In [ ]:
# TODO 6.2 : créer admin, alice et bob en SCRAM-SHA-512 avec 8192 itérations
# 1. Pour chaque compte de ACCOUNTS, construisez
#      UserScramCredentialUpsertion(name, ScramCredentialInfo(ScramMechanism.SCRAM_SHA_512, 8192), password.encode())
#    puis envoyez la liste avec admin.alter_user_scram_credentials([...]) ; attendez chaque futur (result()).
# 2. Lisez les comptes : admin.describe_user_scram_credentials().result() renvoie {name: description} ;
#    description.scram_credential_infos est une liste d'objets avec .mechanism et .iterations.
#    Rangez dans accounts {name: [(nom du mécanisme, itérations), ...]}.
ACCOUNTS = {"admin": "admin-secret", "alice": "alice-secret", "bob": "bob-secret"}
accounts = {}

print(accounts)

In [ ]:
# Vérification 6.2
assert sorted(accounts) == ["admin", "alice", "bob"], "À revoir : 3 comptes : %s" % sorted(accounts)
print("OK  3 comptes : %s" % sorted(accounts))
assert all(infos == [("SCRAM_SHA_512", 8192)] for infos in accounts.values()), "À revoir : SCRAM-SHA-512, 8192 itérations"
print("OK  SCRAM-SHA-512, 8192 itérations")

In [ ]:
!docker exec kafka-tools kafka-configs.sh --bootstrap-server $BS --describe --entity-type users

**Résultat attendu** : 3 comptes en `SCRAM-SHA-512` avec 8192 itérations. Les mots de passe ne sont jamais affichés, ni en Python ni en ligne de commande.

### 6.3 Se connecter en administrateur

Un client Java lit sa configuration de sécurité dans un fichier `.properties`. Le client Python (librdkafka) attend les mêmes informations, sous d'autres noms :

| Java (`.properties`) | Python (librdkafka) |
|---|---|
| `security.protocol=SASL_SSL` | `"security.protocol": "SASL_SSL"` |
| `sasl.mechanism=SCRAM-SHA-512` | `"sasl.mechanism": "SCRAM-SHA-512"` |
| `sasl.jaas.config=... username="u" password="p";` | `"sasl.username": "u"`, `"sasl.password": "p"` |
| `ssl.truststore.location=...truststore.p12` et son mot de passe | `"ssl.ca.location": CA_PEM` (certificat de la CA en PEM) |
| `client.id=...` | `"client.id": "..."` |

**Exemples en ligne de commande** : le fichier du compte `admin`, puis la liste des topics par le listener SECURE avec ce fichier.

In [ ]:
!docker exec kafka-tools cat /opt/lab/clients/admin.properties
!docker exec kafka-tools bash -c "kafka-topics.sh --bootstrap-server $BS_SECURE --command-config /opt/lab/clients/admin.properties --list 2>&1 | tail -12"

In [ ]:
# TODO 6.3 : configurer le client Python du compte admin
# 1. Écrivez secure_config(user, password) : elle renvoie le dictionnaire de configuration
#    d'un client SASL_SSL sur BS_SECURE (5 clés d'après le tableau, plus "bootstrap.servers").
# 2. config_admin = secure_config("admin", "admin-secret") ; rangez connection(config_admin) dans admin_attempt.
# 3. Rangez dans bad_password_attempt le résultat de connection() pour alice avec le mot de passe "mauvais".
def secure_config(user, password):
    return {}


config_admin = secure_config("admin", "admin-secret")
admin_attempt = bad_password_attempt = None

print("admin   :", admin_attempt)
print("mauvais :", bad_password_attempt)

In [ ]:
# Vérification 6.3
assert admin_attempt is not None and admin_attempt[0] == "OK" and "orders" in admin_attempt[1], "À revoir : admin liste les topics par SECURE"
print("OK  admin liste les topics par SECURE")
assert bad_password_attempt is not None and bad_password_attempt[0] == "_AUTHENTICATION", "À revoir : mauvais mot de passe : échec d'authentification"
print("OK  mauvais mot de passe : échec d'authentification")

**Résultat attendu** :

- `admin` liste tous les topics ;
- le mauvais mot de passe donne `_AUTHENTICATION` : `SASL authentication error: Authentication failed during authentication due to invalid credentials with SASL mechanism SCRAM-SHA-512`. Le client Java affiche le même texte dans une `SaslAuthenticationException`.

### 6.4 Autoriser le producteur `alice`

Une ACL associe : un **type de ressource** (topic, groupe, cluster), un **nom** et un **motif** (`LITERAL` pour un nom exact, `PREFIXED` pour un préfixe), un **principal** (`User:alice`), un **hôte** (`*` pour tous), une **opération** (`WRITE`, `READ`, `DESCRIBE`...) et une **permission** (`ALLOW` ou `DENY`).

**Exemple en ligne de commande** : les refus d'`alice` sont journalisés par le broker qui a traité la requête, dans `kafka-authorizer.log` (fichier archivé toutes les heures : la commande relit les archives dans l'ordre, puis garde la dernière ligne de chaque nœud). Avant votre premier essai, elle n'affiche rien. Les heures des journaux sont en UTC, l'heure des conteneurs.

In [ ]:
!for n in 1 2 3; do docker compose exec -T kafka-$$n sh -c 'cd /opt/kafka/logs && cat $$(ls -tr kafka-authorizer.log*) | grep "User:alice is Denied" | tail -1' </dev/null; done

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-acls.sh --bootstrap-server $BS --add --allow-principal User:alice \
  --operation Write --operation Describe --topic orders
```

In [ ]:
# TODO 6.4 : faire écrire alice dans orders, avant puis après les ACL
# 1. config_alice : secure_config() d'alice, plus "client.id": "orders-service".
# 2. Rangez dans before_acl le résultat de send(config_alice, "orders", '{"order_id":"O-5001"}', key="C010").
# 3. Exécutez la cellule d'exemple ci-dessus pour retrouver le refus dans kafka-authorizer.log.
# 4. Créez les ACL WRITE et DESCRIBE d'alice sur le topic orders : admin.create_acls([AclBinding(...), ...]),
#    avec AclBinding(ResourceType.TOPIC, "orders", ResourcePatternType.LITERAL, "User:alice", "*",
#                    AclOperation.WRITE, AclPermissionType.ALLOW) ; attendez chaque futur.
# 5. Rangez dans after_acl le résultat du même envoi.
config_alice = {}
before_acl = after_acl = "à faire"

print("avant les ACL :", before_acl, "- après :", after_acl)

In [ ]:
# Vérification 6.4
assert before_acl == "TOPIC_AUTHORIZATION_FAILED", "À revoir : sans ACL, alice est refusée : %s" % before_acl
print("OK  sans ACL, alice est refusée : %s" % before_acl)
assert after_acl is None, "À revoir : avec les ACL, alice écrit dans orders"
print("OK  avec les ACL, alice écrit dans orders")

In [ ]:
!for n in 1 2 3; do docker compose exec -T kafka-$$n sh -c 'cd /opt/kafka/logs && cat $$(ls -tr kafka-authorizer.log*) | grep "User:alice is Denied" | tail -1' </dev/null; done

**Résultat attendu** :

- sans ACL : `TOPIC_AUTHORIZATION_FAILED` ;
- dans `kafka-authorizer.log` : `Principal = User:alice is Denied operation = DESCRIBE ... on resource = Topic:LITERAL:orders for request = Metadata ... based on rule DefaultDeny` ;
- après les ACL : envoi sans erreur (`None`).

### 6.5 Défi : écrire la configuration de `bob`

**Exemple en ligne de commande** : le modèle de configuration Java, à compléter dans la version Markdown.

In [ ]:
!docker exec kafka-tools cat /opt/lab/clients/secure-template.properties

In [ ]:
# TODO 6.5 (défi) : config_bob, la configuration Python du compte bob (client.id billing-service),
# sans appeler secure_config() : écrivez le dictionnaire complet, à partir du tableau de l'étape 6.3.
config_bob = {}

print(sorted(config_bob))

In [ ]:
# Vérification 6.5
assert config_bob.get("client.id") == "billing-service", "À revoir : client.id de bob"
print("OK  client.id de bob")
assert connection(config_bob)[0] == "OK", "À revoir : bob s'authentifie sur le listener SECURE"
print("OK  bob s'authentifie sur le listener SECURE")

**Résultat attendu** : `bob` s'authentifie. Un `client.id` est facultatif pour la connexion, mais il sert aux quotas, aux métriques et aux journaux.

### 6.6 Autoriser le consommateur `bob`

Un consommateur de groupe a besoin de deux droits : lire le **topic** et utiliser le **groupe**.

Équivalent en ligne de commande des ACL du `TODO` (**ne pas lancer en plus du TODO**) : une seule commande crée les ACL sur les deux ressources.

```bash
# [tools]
kafka-acls.sh --bootstrap-server $BS --add --allow-principal User:bob \
  --operation Read --operation Describe --topic orders --group billing-secure
```

In [ ]:
# TODO 6.6 : lecture et écriture de bob
# 1. read_without_acl : read(config_bob, "billing-secure").
# 2. Créez 4 ACL ALLOW pour User:bob : READ et DESCRIBE sur le topic orders, READ et DESCRIBE sur le groupe
#    billing-secure (ResourceType.GROUP).
# 3. read_with_acl : read(config_bob, "billing-secure") ; analytics_read : read(config_bob, "analytics").
# 4. bob_write : send(config_bob, "orders", '{"order_id":"O-5002"}', key="C011").
#    bob_idempotent_write : le même envoi, avec en plus le réglage enable.idempotence à True. Son nom
#    contient des points : passez-le par un dictionnaire, send(..., **{"enable.idempotence": True}).
read_without_acl = read_with_acl = analytics_read = None
bob_write = bob_idempotent_write = "à faire"

table([("lecture, billing-secure, sans ACL", read_without_acl),
       ("lecture, billing-secure, avec ACL", read_with_acl),
         ("lecture, analytics", analytics_read),
         ("écriture dans orders", bob_write),
         ("écriture idempotente dans orders", bob_idempotent_write)], ["Essai de bob", "Résultat"])

In [ ]:
# Vérification 6.6
assert read_without_acl in ("GROUP_AUTHORIZATION_FAILED", "TOPIC_AUTHORIZATION_FAILED"), "À revoir : sans ACL, bob est refusé : %s" % read_without_acl
print("OK  sans ACL, bob est refusé : %s" % read_without_acl)
assert isinstance(read_with_acl, list) and len(read_with_acl) == 5, "À revoir : avec les ACL, bob lit 5 commandes"
print("OK  avec les ACL, bob lit 5 commandes")
assert analytics_read == "GROUP_AUTHORIZATION_FAILED", "À revoir : groupe analytics refusé"
print("OK  groupe analytics refusé")
assert bob_write == "TOPIC_AUTHORIZATION_FAILED", "À revoir : écriture refusée sur le topic"
print("OK  écriture refusée sur le topic")
assert bob_idempotent_write == "CLUSTER_AUTHORIZATION_FAILED", "À revoir : écriture idempotente refusée sur le cluster"
print("OK  écriture idempotente refusée sur le cluster")

**Résultat attendu** :

| Essai | Python (librdkafka) | Java (version Markdown) |
|---|---|---|
| Lecture sans ACL | `GROUP_AUTHORIZATION_FAILED` | `TopicAuthorizationException` |
| Lecture avec ACL, groupe `billing-secure` | 5 commandes | 5 commandes |
| Lecture avec le groupe `analytics` | `GROUP_AUTHORIZATION_FAILED` | `GroupAuthorizationException` |
| Écriture | `TOPIC_AUTHORIZATION_FAILED` | `ClusterAuthorizationException` |
| Écriture idempotente | `CLUSTER_AUTHORIZATION_FAILED` | (défaut du client Java) |

### 6.7 Auditer les ACL

**Exemple en ligne de commande** : les ACL du topic `orders`.

In [ ]:
!docker exec kafka-tools kafka-acls.sh --bootstrap-server $BS --list --topic orders

In [ ]:
# TODO 6.7 : les mêmes listes en Python
# admin.describe_acls(filtre).result() renvoie la liste des AclBinding qui correspondent au filtre :
#   AclBindingFilter(type de ressource, nom, motif, principal, hôte, opération, permission)
# None vaut "n'importe lequel" pour le nom, le principal et l'hôte ; ResourceType.ANY, ResourcePatternType.ANY,
# AclOperation.ANY et AclPermissionType.ANY pour les autres.
# 1. acl_orders : liste triée des (principal, nom de l'opération) des ACL du topic orders (motif LITERAL).
# 2. acl_bob : liste triée des (nom du type de ressource, nom, nom de l'opération) des ACL de User:bob.
acl_orders = acl_bob = []

print(acl_orders)
print(acl_bob)

In [ ]:
# Vérification 6.7
assert (acl_orders == [("User:alice", "DESCRIBE"), ("User:alice", "WRITE"),
                        ("User:bob", "DESCRIBE"), ("User:bob", "READ")]), "À revoir : 4 ACL sur orders"
print("OK  4 ACL sur orders")
assert len(acl_bob) == 4 and {r for r, _, _ in acl_bob} == {"TOPIC", "GROUP"}, "À revoir : bob : 2 ACL de topic, 2 de groupe"
print("OK  bob : 2 ACL de topic, 2 de groupe")

**Résultat attendu** : le topic `orders` porte 4 ACL : `WRITE` et `DESCRIBE` pour `alice`, `READ` et `DESCRIBE` pour `bob`. `bob` a 2 ACL sur le topic et 2 sur le groupe `billing-secure`.

### 6.8 Quota de production

Un quota `producer_byte_rate` limite le débit d'écriture d'un principal (ou d'un `client.id`). Le client Python ne sait pas poser de quota : on passe par l'outil Java, avec une cellule `!docker exec kafka-tools kafka-configs.sh ...`.

La fonction `measure()` ci-dessous envoie `count` messages de `size` octets dans `quota-test`, le plus vite possible, et renvoie le débit, la durée et les **freinages** reçus du broker (`throttle_cb` : appelé à chaque réponse qui demande au client d'attendre).

Deux réglages rapprochent le producteur Python du test de performance Java : des lots de 16 Ko (`batch.size`), et une partition tirée au hasard pour chaque message (`sticky.partitioning.linger.ms` à 0). Sans eux, librdkafka envoie de gros lots vers une seule partition à la fois, donc vers un seul broker.

In [ ]:
def measure(config, count=8000, size=1000):
    throttles = []
    producer = Producer({**config, "batch.size": 16384, "sticky.partitioning.linger.ms": 0, "log_level": 4,
                         "throttle_cb": lambda event: throttles.append(event.throttle_time)})
    value = b"x" * size
    start = time.time()
    for _ in range(count):
        while True:
            try:
                producer.produce("quota-test", value=value)
                break
            except BufferError:
                # File locale pleine : laisser partir des messages
                producer.poll(0.1)
        producer.poll(0)
    producer.flush(120)
    duration = time.time() - start
    return {"débit Mo/s": round(count * size / duration / 1e6, 2), "durée s": round(duration, 1),
            "freinages": len(throttles), "frein moyen ms": round(1000 * sum(throttles) / len(throttles)) if throttles else 0}

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-topics.sh --bootstrap-server $BS --create --topic quota-test --partitions 3 --replication-factor 3
kafka-acls.sh --bootstrap-server $BS --add --allow-principal User:alice \
  --operation Write --operation Describe --topic quota-test
kafka-configs.sh --bootstrap-server $BS --alter --entity-type users --entity-name alice \
  --add-config producer_byte_rate=102400
kafka-producer-perf-test.sh --bootstrap-server $BS_SECURE --command-config /opt/lab/clients/alice.properties \
  --topic quota-test --num-records 8000 --record-size 1000 --throughput -1 --print-metrics
kafka-configs.sh --bootstrap-server $BS --alter --entity-type users --entity-name alice \
  --delete-config producer_byte_rate
```

In [ ]:
# TODO 6.8 : brider alice et mesurer
# 1. Créez quota-test (3 partitions, RF 3) et donnez à alice WRITE et DESCRIBE sur ce topic (Python).
# 2. Posez le quota producer_byte_rate=102400 sur l'utilisateur alice, avec `!docker exec kafka-tools kafka-configs.sh ...`
#    (--entity-type users --entity-name alice --add-config ...).
# 3. with_quota = measure(config_alice)
# 4. Retirez le quota (--delete-config producer_byte_rate), attendez 2 s, puis without_quota = measure(config_alice)
with_quota = without_quota = {}

table([("avec quota", *with_quota.values()), ("sans quota", *without_quota.values())],
      ["Mesure", "Débit Mo/s", "Durée s", "Freinages", "Frein moyen ms"])

In [ ]:
# Vérification 6.8
assert with_quota and with_quota["débit Mo/s"] < 0.5 and with_quota["freinages"] > 0, "À revoir : avec quota : %s Mo/s, %s freinages" % (with_quota.get("débit Mo/s"), with_quota.get("freinages"))
print("OK  avec quota : %s Mo/s, %s freinages" % (with_quota.get("débit Mo/s"), with_quota.get("freinages")))
assert without_quota and without_quota["débit Mo/s"] > 5 * with_quota.get("débit Mo/s", 1), "À revoir : sans quota : débit bien supérieur"
print("OK  sans quota : débit bien supérieur")
output = !docker exec kafka-tools kafka-configs.sh --bootstrap-server $BS --describe --entity-type users --entity-name alice 2>&1
assert ("producer_byte_rate" not in output.n), "À revoir : quota retiré"
print("OK  quota retiré")

**Résultat attendu** :

| Mesure | Avec quota | Sans quota |
|---|---|---|
| Débit | environ 0,3 Mo/s | plusieurs Mo/s |
| Freinages | plusieurs dizaines, de quelques centaines de millisecondes | aucun |

### 6.9 Matrice des droits

Complétez la matrice avec `"oui"` ou `"non"`, d'après les résultats des étapes précédentes. La vérification la compare aux décisions qu'appliquerait l'authorizer, calculées à partir des ACL réelles du cluster.

In [ ]:
# TODO 6.9 : la matrice des droits
matrix = {
    "Lister les topics": {"admin": "", "alice": "", "bob": ""},
    "Écrire dans orders": {"admin": "", "alice": "", "bob": ""},
    "Lire orders avec le groupe billing-secure": {"admin": "", "alice": "", "bob": ""},
    "Lire orders avec le groupe analytics": {"admin": "", "alice": "", "bob": ""},
    "Écrire dans quota-test": {"admin": "", "alice": "", "bob": ""},
}

table([(action, d["admin"], d["alice"], d["bob"]) for action, d in matrix.items()], ["Action", "admin", "alice", "bob"])

In [ ]:
# Vérification 6.9
# Décision de l'authorizer, recalculée à partir des ACL du cluster et des super-utilisateurs du lab
admin = AdminClient(CONF)
SUPER_USERS = {"User:admin", "User:ANONYMOUS"}
all_acls = admin.describe_acls(AclBindingFilter(ResourceType.ANY, None, ResourcePatternType.ANY, None, None,
                                                    AclOperation.ANY, AclPermissionType.ANY)).result()


def allowed(principal, operation, resource_type, name):
    if principal in SUPER_USERS:
        return True
    rules = [a for a in all_acls if a.principal == principal and a.restype == resource_type
             and a.operation in (operation, AclOperation.ALL)
              and (a.name == name if a.resource_pattern_type == ResourcePatternType.LITERAL else name.startswith(a.name))]
    # Un DENY l'emporte toujours sur un ALLOW
    return any(a.permission_type == AclPermissionType.ALLOW for a in rules) and \
        not any(a.permission_type == AclPermissionType.DENY for a in rules)


def read_with(principal, group):
    return allowed(principal, AclOperation.READ, ResourceType.TOPIC, "orders") and \
        allowed(principal, AclOperation.READ, ResourceType.GROUP, group)


decisions = {
    "Lister les topics": lambda p: True,
    "Écrire dans orders": lambda p: allowed(p, AclOperation.WRITE, ResourceType.TOPIC, "orders"),
    "Lire orders avec le groupe billing-secure": lambda p: read_with(p, "billing-secure"),
    "Lire orders avec le groupe analytics": lambda p: read_with(p, "analytics"),
    "Écrire dans quota-test": lambda p: allowed(p, AclOperation.WRITE, ResourceType.TOPIC, "quota-test"),
}
gaps = [(action, account) for action, decision in decisions.items() for account in ("admin", "alice", "bob")
        if matrix[action][account] != ("oui" if decision("User:" + account) else "non")]
assert not gaps, "À revoir : matrice conforme aux ACL du cluster" + (" ; à revoir : %s" % gaps if gaps else "")
print("OK  matrice conforme aux ACL du cluster" + (" ; à revoir : %s" % gaps if gaps else ""))

**Résultat attendu** : la matrice est conforme. Tout compte authentifié peut lister les topics, mais ne voit que ceux qu'il a le droit de décrire.

## 7. Vérification d'ensemble

**Exemple en ligne de commande** : toutes les ACL et tous les comptes.

In [ ]:
!docker exec kafka-tools kafka-acls.sh --bootstrap-server $BS --list | grep -c permissionType
!docker exec kafka-tools kafka-configs.sh --bootstrap-server $BS --describe --entity-type users

Contrôle : 3 comptes SCRAM, aucun quota restant pour `alice`, 8 ACL conformes à la matrice.

## 8. Questions de réflexion

1. Pourquoi le client Java obtient-il une erreur de mémoire plutôt qu'un message clair ? Et le client Python ?
2. Où sont stockés les comptes SCRAM en mode KRaft ? Qui peut les lire ?
3. Pourquoi `alice` a-t-elle besoin de `Describe` en plus de `Write` ?
4. Pourquoi l'écriture de `bob` est-elle refusée sur le **cluster** avec un producteur idempotent, et sur le topic sans ?
5. Le quota de 100 Ko/s est-il global ou par broker ? Que montre le débit mesuré ?
6. Quels sont les deux écarts de sécurité de ce lab qui seraient inacceptables en production ?
7. Comment renouveler le mot de passe de `alice` sans interruption de service ?